# ¿Qué hace que un juego en Steam sea popular?

## 1. Introducción

### 1.1 Contexto

### 1.2 Objetivo

### 1.3 Variable Dependiente

### 1.4 Variable Independiente

### 1.5 Setup

In [ ]:
import os

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline # no es necesario

RANDOM_STATE = 2026
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 2. Base de Datos

### 2.1 Fuente

Martin Bustos Roman. (2022). Steam Games Dataset [Dataset]. Kaggle. https://doi.org/10.34740/KAGGLE/DS/2109585

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("fronkongames/steam-games-dataset")
print("Path to dataset files:", path)

path_to_csv = os.path.join(path, "games.csv")
print("Path to csv file:", path_to_csv)

### 2.2 Tamaño

In [ ]:
# Loading the dataset and having a glimpse
raw_dataset = pd.read_csv(path_to_csv,
                   na_values="?",
                   comment='\t',
                   sep=','
                   )
print(f"Dimensions: {raw_dataset.shape}")

In [ ]:
# Taking a look at the head of the dataset
raw_dataset.head()

In [ ]:
# From a quick look and reading the discussion section in kaggle
# we notice there seems to be a comma missing between the header
# Discount, DLC count
# Applying the fix...
header = raw_dataset.columns.to_list()
print(f"Old Header: {header}")

fixed_header = header[:7] + ["Discount", "DLC count"] + header[8:]
fixed_dataset = pd.read_csv(path_to_csv,
                            na_values="?",
                            comment='\t',
                            sep=',',
                            names=fixed_header,
                            header=0
                            )
print(f"Fixed Dimensions: {fixed_dataset.shape}")
fixed_dataset.head()

### 2.3 Análisis

In [ ]:
# Initial statistics of the data
fixed_dataset.describe().T

falta texto, notar que es analisis de la base de datos, en general

### 2.4 Diccionario de Variables

Antes de empezar todo el analisis que viene a continuación, fue necesario corregir el header, pues dos variables venian unidas, pues le faltaba una coma.

Esto puede ser corroborado arriba.

- DiscountDLC count -> Discount, DLC count

De esta manera tendríamos las 40 variables iniciales.

#### 2.4.1 Tipos de Dato

In [ ]:
# Checking the data
fixed_dataset.info()

#### 2.4.2 Significado

significado de las variables...


## 3. Limpieza y Preparación

### 3.1 Suposiciones de Data

#### 3.1.1 Missing Values

Antes de avanzar debemos de revisar cuanta de la data es vacia (NA) y cuanta de ella es realmente NA disfrazada, por ejemplo, "[]" y ceros en algunas columnas.

Hay que tener cuidado de no tomar cualquier 0 como NA, por ejemplo, precio 0 quiere decir gratis y no que este vacio.

In [ ]:
def missing_summary_table(df):
    """
    Returns a summary table of missing values in the dataset.
    """
    # Each measure is a Series: index = column name, value = % of rows
    numeric_cols = df.select_dtypes(include="number")
    zero_pct = (numeric_cols == 0).mean() * 100

    null_pct = df.isna().mean() * 100

    empty_list_pct = df.isin(["[]"]).mean() * 100

    summary = pd.DataFrame({
        "% nulos": null_pct,
        "% lista vacía '[]'": empty_list_pct,
        "% ceros": zero_pct,
    })

    summary["% vacío total"] = summary["% nulos"] + summary["% lista vacía '[]'"]
    return summary.sort_values(["% vacío total", "% ceros"], ascending=False).round(2)

missing_summary_table(fixed_dataset)

In [ ]:
# Check: is "Metacritic score == 0" the same as "no Metacritic URL"?
score_is_zero = fixed_dataset["Metacritic score"] == 0
url_is_missing = fixed_dataset["Metacritic url"].isna()

pd.crosstab(
    score_is_zero,
    url_is_missing,
    rownames=["Metacritic score = 0"],
    colnames=["Metacritic url vacío"],
)

- Al analizar los datos vacios podemos decir que definitivamente la columna Movies se puede botar, pues no contiene nada de data. 
- Otro candidato para ser botado es Score rank.
- Metacritic score = 0 significa "sin reseña de crítica", no una calificación de 0.

Otras suposiciones:
- Price 0 = gratis
- Required age 0 = todo el publico.

#### 3.1.2 Juegos sin Actividad

In [ ]:
# Check: do games with "0 - 0" owners have any reviews?
no_owners = fixed_dataset["Estimated owners"] == "0 - 0"
games_with_reviews = fixed_dataset["Positive"] + fixed_dataset["Negative"] > 0

pd.crosstab(
    no_owners,
    games_with_reviews,
    rownames=["Owners = 0 - 0"],
    colnames=["Juegos con reviews"],
)

De aquí se podría decir que 24579 son juegos sin actividad registrada (playtests, próximos lanzamientos o sin datos), por lo que no aporta a la pregunta de que determina la popularidad de un juego. Por lo tanto, estos juegos pueden ser botados.

### 3.2 Modificaciones

#### 3.2.1 Juegos sin Actividad

In [ ]:
# Keep only games with registered activity (owners other than "0 - 0")
has_activity = fixed_dataset["Estimated owners"] != "0 - 0"
released_games = fixed_dataset[has_activity].copy()

print(f"Antes:      {fixed_dataset.shape}")
print(f"Después:    {released_games.shape}")
print(f"Eliminados: {len(fixed_dataset) - len(released_games):,} juegos")

In [ ]:
# Compare % empty per column before and after removing inactive games
before = missing_summary_table(fixed_dataset)["% vacío total"]
after = missing_summary_table(released_games)["% vacío total"]

empty_comparison = pd.DataFrame({
    "% vacío antes": before,
    "% vacío después": after,
})
empty_comparison["diferencia"] = empty_comparison["% vacío después"] - empty_comparison["% vacío antes"]

# Only columns that had something empty, largest first
empty_comparison[empty_comparison["% vacío antes"] > 0].sort_values("% vacío antes", ascending=False)

Solo quitando aquellos juegos sin actividad se logro limpiar bastantes los datos vacios. Ahora un ultimo paso...

## 4. Análisis Descriptivo y Correlación

### 4.1 Estadísticas Relevantes

### 4.2 Matriz de Correlación

### 4.3 Interpretración

## 5. Selección de Variables

método completo fundamentado hasta el modelo final

## 6. Train/Test Split

análisis completo de los resultados del modelo final

## 7. Modelo Final (Regresión Lineal)

### 7.1 Ecuación Estimada

### 7.2 Interpretración de Coeficientes

## 8. Modelos Extra

Lasso, Ridge

No lineales:
- Random Forest
- Gradient Boosting
- Suport Vector Regression (SVR)

## 9. Conclusiones

### 9.1 Hallazgos

### 9.2 Limitaciones

### 9.3 Recomendaciones

### 9.4 Respuesta al Objetivo

## 10. Referencias

código (puede ser un link), tablas complementarias, etc.